# Лабораторна робота 3. Машинне навчання
## Регресія: прогнозування вартості нерухомості

**Мета роботи:** виконати первинний аналіз даних, побудувати кілька моделей регресії, оцінити їхню якість та виконати підбір гіперпараметрів для моделі Random Forest.

### Датасет
Можна використати один із двох способів завантаження даних:

1. Завантажити файл з Google Drive:
https://drive.google.com/file/d/1u_-3X05st6uvrgQB1qNLyE8a7QjTJjvd/view?usp=sharing

2. Завантажити датасет з Kaggle:
https://www.kaggle.com/datasets/prokshitha/home-value-insights

> Посилання та вміст датасету на зовнішніх платформах можуть змінюватися.
**Виконав:** Харченко Артьом, група ФІТ 3-10

## 1. Імпорт бібліотек
Імпортуйте бібліотеки, необхідні для роботи з даними, візуалізації та числових обчислень.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# щоб великі числа в таблицях виводились без e+08
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## 2. Завантаження даних
Завантажте файл двома способами.

### Варіант 1. Завантаження файлу з комп’ютера
Завантажте CSV-файл у середовище Colab.

In [ ]:
from google.colab import files

# після запуску з'явиться кнопка вибору файлу, треба вибрати csv з комп'ютера
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
df.head()

### Варіант 2. Завантаження датасету з Kaggle
Завантажте датасет безпосередньо з Kaggle та прочитайте CSV-файл у DataFrame `df`.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("prokshitha/home-value-insights")
print(os.listdir(path))

df_kaggle = pd.read_csv(os.path.join(path, "house_price_regression_dataset.csv"))
print("Дані з Kaggle збігаються з файлом з комп'ютера:", df.equals(df_kaggle))

df = df_kaggle
df.head()

## 3. Опис ознак датасету

- **Square_Footage** — загальна житлова площа будинку.
- **Num_Bedrooms** — кількість спалень.
- **Num_Bathrooms** — кількість ванних кімнат.
- **Year_Built** — рік побудови будинку.
- **Lot_Size** — розмір земельної ділянки.
- **Garage_Size** — розмір гаража або кількість паркомісць.
- **Neighborhood_Quality** — оцінка якості району.
- **House_Price** — ціна будинку, цільова змінна.

## 4. Первинний аналіз даних

### Завдання 1. Перегляд даних
Виведіть перші рядки датасету та перегляньте його структуру.

In [ ]:
display(df.head(10))
df.info()

### Завдання 2. Перевірка дублікатів
Визначте кількість дубльованих рядків.

In [ ]:
print("Кількість дублікатів:", df.duplicated().sum())

### Завдання 3. Перевірка пропущених значень
Визначте кількість пропущених значень у кожному стовпці.

In [ ]:
df.isna().sum()

### Завдання 4. Описова статистика
Виведіть основні статистичні характеристики числових ознак.

In [ ]:
df.describe()

## 5. Аналіз окремих ознак

### Завдання 5. Ознака `Neighborhood_Quality`
Виведіть частоти значень ознаки `Neighborhood_Quality`.

In [ ]:
df["Neighborhood_Quality"].value_counts().sort_index()

### Завдання 6. Візуалізація `Neighborhood_Quality`
Побудуйте діаграму розподілу значень цієї ознаки.

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x="Neighborhood_Quality", data=df)
plt.title("Розподіл оцінки якості району")
plt.xlabel("Neighborhood_Quality")
plt.ylabel("Кількість будинків")
plt.show()

### Завдання 7. Ознака `Num_Bathrooms`
Виведіть частоти значень кількості ванних кімнат.

In [ ]:
df["Num_Bathrooms"].value_counts().sort_index()

### Завдання 8. Візуалізація `Num_Bathrooms`
Побудуйте діаграму розподілу кількості ванних кімнат.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="Num_Bathrooms", data=df)
plt.title("Розподіл кількості ванних кімнат")
plt.xlabel("Num_Bathrooms")
plt.ylabel("Кількість будинків")
plt.show()

### Додаткове завдання
За потреби побудуйте попарні графіки числових ознак і коротко проаналізуйте можливі залежності.

In [ ]:
# беремо тільки основні числові ознаки, щоб графік було легше читати
sns.pairplot(df[["Square_Footage", "Lot_Size", "Year_Built", "House_Price"]], plot_kws={"s": 10})
plt.show()

**Аналіз:** на графіках видно, що ціна будинку майже лінійно залежить від площі (Square_Footage), точки лежать майже на прямій. З розміром ділянки є слабка залежність, а від року побудови ціна майже не залежить. Самі ознаки розподілені приблизно рівномірно.

## 6. Кореляційний аналіз

### Завдання 9. Матриця кореляції
Побудуйте кореляційну матрицю для числових ознак та відобразіть її у вигляді теплової карти.

In [ ]:
corr = df.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Матриця кореляції")
plt.show()

### Завдання 10. Кореляція з цільовою змінною
Виведіть кореляцію кожної ознаки з `House_Price`, відсортувавши значення за спаданням.

In [ ]:
corr["House_Price"].drop("House_Price").sort_values(ascending=False)

Найсильніше з ціною корелює площа будинку (0.99). Далі розмір ділянки (0.16), у решти ознак кореляція майже нульова.

## 7. Побудова моделей регресії

### Завдання 11. Імпорт інструментів Scikit-learn
Імпортуйте засоби для:
- поділу вибірки;
- масштабування ознак;
- побудови Linear Regression, Ridge, Lasso, Random Forest та Gradient Boosting;
- створення Pipeline;
- обчислення R², MAE та MSE.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

### Завдання 12. Формування `X` і `y`
Відокремте ознаки від цільової змінної `House_Price`.

In [ ]:
X = df.drop(columns=["House_Price"])
y = df["House_Price"]

print("X:", X.shape)
print("y:", y.shape)

### Завдання 13. Поділ даних
Поділіть дані на навчальну і тестову вибірки у співвідношенні 80/20. Для відтворюваності використайте `random_state=42`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Навчальна вибірка:", X_train.shape)
print("Тестова вибірка:", X_test.shape)

### Завдання 14. Створення моделей
Створіть п’ять моделей:
1. Linear Regression;
2. Ridge;
3. Lasso;
4. Random Forest Regressor;
5. Gradient Boosting Regressor.

Для лінійних моделей використайте `Pipeline` зі `StandardScaler`. Для деревоподібних моделей масштабування не потрібне.

In [ ]:
models = {
    "Linear Regression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso())]),
    # деревам масштабування не потрібне
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

### Завдання 15. Навчання та оцінювання моделей
Для кожної моделі:
- виконайте навчання на тренувальній вибірці;
- отримайте прогноз для тестової вибірки;
- обчисліть `R²`, `MAE`, `MSE`;
- збережіть результати для подальшого порівняння.

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    results.append({
        "Модель": name,
        "R2": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
    })

results = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
results

### Завдання 16. Графічне порівняння прогнозів
Для кожної моделі побудуйте графік **фактичні значення vs прогнозовані значення**. Додайте лінію ідеального прогнозу.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[i]
    ax.scatter(y_test, y_pred, s=12, alpha=0.6)
    # лінія ідеального прогнозу: прогноз = факт
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red")
    ax.set_title(name)
    ax.set_xlabel("Фактична ціна")
    ax.set_ylabel("Прогнозована ціна")

axes[5].axis("off")  # шоста клітинка сітки не потрібна
plt.tight_layout()
plt.show()

## 8. Підбір гіперпараметрів Random Forest

### Завдання 17. GridSearchCV
Виконайте підбір оптимальних параметрів для `RandomForestRegressor` за допомогою `GridSearchCV`.

Дослідіть щонайменше такі параметри:
- `n_estimators`;
- `max_features`;
- `max_depth`;
- `min_samples_split`.

Використайте крос-валідацію та метрику `R²`.

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_features": [1.0, "sqrt"],
    "max_depth": [None, 10],
    "min_samples_split": [2, 5],
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print("Найкращі параметри:", grid.best_params_)
print("Найкращий R2 на крос-валідації:", round(grid.best_score_, 4))

### Завдання 18. Оптимізована модель Random Forest
Створіть нову модель з найкращими параметрами, навчіть її та обчисліть `R²`, `MAE`, `MSE` на тестовій вибірці.

In [ ]:
rf_best = RandomForestRegressor(**grid.best_params_, random_state=42)
rf_best.fit(X_train, y_train)
y_pred_best = rf_best.predict(X_test)

print("R2:", round(r2_score(y_test, y_pred_best), 4))
print("MAE:", round(mean_absolute_error(y_test, y_pred_best), 2))
print("MSE:", round(mean_squared_error(y_test, y_pred_best), 2))

### Завдання 19. Порівняння Random Forest до і після оптимізації
Порівняйте значення `R²`, `MAE` і `MSE` для базової та оптимізованої моделей.

In [ ]:
y_pred_rf = predictions["Random Forest"]

compare = pd.DataFrame({
    "До оптимізації": [r2_score(y_test, y_pred_rf), mean_absolute_error(y_test, y_pred_rf), mean_squared_error(y_test, y_pred_rf)],
    "Після оптимізації": [r2_score(y_test, y_pred_best), mean_absolute_error(y_test, y_pred_best), mean_squared_error(y_test, y_pred_best)],
}, index=["R2", "MAE", "MSE"])

compare["Різниця"] = compare["Після оптимізації"] - compare["До оптимізації"]
compare

### Завдання 20. Аналіз оптимізації
Зробіть короткий висновок:
- чи покращилася якість моделі після підбору параметрів;
- які метрики змінилися;
- чи доцільно використовувати оптимізовану модель.

**Ваш висновок:**

Після підбору параметрів якість Random Forest майже не змінилась. R2 виріс з 0.9939 до 0.9940, MAE зменшилась приблизно на 180 (з 16,1 тис. до 15,9 тис.), MSE теж трохи зменшилась. Тобто покращились всі три метрики, але дуже мало.

Найкращими виявились параметри, які майже збігаються зі стандартними, тільки дерев більше (300 замість 100), тому і різниця маленька.

Оптимізовану модель використовувати можна, бо вона трохи краща за базову, але великого сенсу в цьому немає: виграш мінімальний, а навчається вона довше. До того ж на цих даних лінійні моделі все одно точніші за Random Forest.

## 9. Аналіз прогнозів лінійної регресії

### Завдання 21. Таблиця прогнозів
Для моделі Linear Regression створіть таблицю з трьома стовпцями:
- фактична ціна;
- прогнозована ціна;
- похибка у відсотках.

Виведіть 10 випадкових рядків.

In [ ]:
lr_table = pd.DataFrame({
    "Фактична ціна": y_test.values,
    "Прогнозована ціна": predictions["Linear Regression"],
})
lr_table["Похибка, %"] = abs(lr_table["Фактична ціна"] - lr_table["Прогнозована ціна"]) / lr_table["Фактична ціна"] * 100

print("Середня похибка:", round(lr_table["Похибка, %"].mean(), 2), "%")
print("Максимальна похибка:", round(lr_table["Похибка, %"].max(), 2), "%")
lr_table.sample(10, random_state=42)

## 10. Висновки
Сформулюйте загальний висновок за результатами лабораторної роботи. Вкажіть:

1. Які моделі було побудовано.
2. Яка модель показала найкращі значення метрик.
3. Як вплинув підбір гіперпараметрів Random Forest.
4. Наскільки точними є прогнози на тестовій вибірці.
5. Які фактори можуть впливати на якість прогнозування ціни нерухомості.

**Загальний висновок:**

1. Було побудовано 5 моделей регресії: Linear Regression, Ridge і Lasso (з масштабуванням StandardScaler через Pipeline), Random Forest і Gradient Boosting.
2. Найкращі метрики показали лінійні моделі. У Linear Regression R2 = 0.9984 і MAE близько 8,2 тис., Lasso дала майже такий самий результат, Ridge трохи гірший. Далі Gradient Boosting (R2 = 0.9965) і найгірше Random Forest (R2 = 0.9939).
3. Підбір гіперпараметрів Random Forest через GridSearchCV покращив модель зовсім трохи (R2 з 0.9939 до 0.9940), і вона все одно залишилась гіршою за лінійні моделі.
4. Прогнози на тестовій вибірці дуже точні. У лінійної регресії середня похибка 1,66%, а максимальна приблизно 8,5%.
5. На ціну найбільше впливає площа будинку, у неї кореляція з ціною 0.99, решта ознак впливає слабо. Схоже, що дані штучні і ціна в них майже лінійно залежить від ознак, тому лінійна регресія і працює найкраще. У реальних даних на ціну ще впливають розташування, стан будинку, ситуація на ринку, а також викиди і пропуски в даних, тому там прогнозувати було б складніше.